# Chapter 2: Async Batch Email Personalisation

**From *Mastering Agentic AI for Marketing Technology* by Pushparajan Ramar**

---

Personalising emails at scale is one of the most impactful applications of LLMs in martech.
But calling an LLM once per contact in a synchronous loop is painfully slow — a 500-contact
batch would take 15–25 minutes if each call averages 2–3 seconds.

This notebook demonstrates the **asyncio + httpx** pattern for making parallel LLM calls,
reducing that wall-clock time by 10–20×.

### What you will learn

1. How to build a mock contact list that resembles real CRM exports.
2. How to structure an async LLM caller using `httpx.AsyncClient`.
3. How to throttle concurrency with `asyncio.Semaphore` to respect rate limits.
4. How to collect and inspect results for a 500-contact batch.

## 1 — Environment Setup

In [ ]:
import os
import json
import time
import random
import asyncio
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Any, Optional

from dotenv import load_dotenv

load_dotenv()

USE_MOCK = os.getenv("USE_MOCK_APIS", "true").lower() == "true"

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
LLM_BASE_URL = "https://api.openai.com/v1/chat/completions"
LLM_MODEL = "gpt-4.1"

print(f"USE_MOCK = {USE_MOCK}")
print(f"API key present: {bool(OPENAI_API_KEY)}")

## 2 — Generate a Mock Contact List

We create 500 synthetic contacts with attributes you would typically pull from a CRM:
name, company, industry, role, last engagement, and lifecycle stage.

In [ ]:
# --- Mock contact generator ---

FIRST_NAMES = [
    "Alice", "Bob", "Carla", "David", "Elena", "Frank", "Grace", "Hiro",
    "Ines", "Jake", "Kara", "Liam", "Mona", "Nate", "Olivia", "Priya",
    "Quinn", "Raj", "Sara", "Tom", "Uma", "Vic", "Wendy", "Xander", "Yuki", "Zara",
]

LAST_NAMES = [
    "Smith", "Johnson", "Lee", "Patel", "Garcia", "Chen", "Kim", "Nguyen",
    "Brown", "Martinez", "Kumar", "Taylor", "Wilson", "Anderson", "Thomas",
    "Jackson", "White", "Harris", "Clark", "Lewis",
]

COMPANIES = [
    "Acme SaaS", "Beacon Analytics", "CloudSync", "DataPipe", "Elevate CRM",
    "FlowMetrics", "GrowthLab", "HyperLead", "InsightIQ", "JoltStack",
    "KineticOps", "LeadForge", "MetricWave", "NovaPlatform", "OrbitAI",
]

INDUSTRIES = [
    "SaaS", "FinTech", "HealthTech", "E-Commerce", "EdTech",
    "MarTech", "HRTech", "Cybersecurity", "DevTools", "Logistics",
]

ROLES = [
    "VP of Marketing", "Director of Growth", "CMO", "Head of Demand Gen",
    "Marketing Manager", "Director of Marketing", "Growth Lead",
    "Head of Content", "Director of Digital", "VP of Revenue Marketing",
]

LIFECYCLE_STAGES = ["subscriber", "lead", "mql", "sql", "opportunity", "customer"]

LAST_ENGAGEMENTS = [
    "downloaded whitepaper", "attended webinar", "visited pricing page",
    "opened last email", "clicked CTA in blog post", "requested demo",
    "filled out contact form", "viewed case study",
]


@dataclass
class Contact:
    id: int
    first_name: str
    last_name: str
    email: str
    company: str
    industry: str
    role: str
    lifecycle_stage: str
    last_engagement: str


def generate_contacts(n: int = 500) -> List[Contact]:
    """Generate n synthetic CRM contacts."""
    contacts = []
    for i in range(n):
        first = random.choice(FIRST_NAMES)
        last = random.choice(LAST_NAMES)
        company = random.choice(COMPANIES)
        domain = company.lower().replace(" ", "") + ".com"
        contacts.append(Contact(
            id=i + 1,
            first_name=first,
            last_name=last,
            email=f"{first.lower()}.{last.lower()}@{domain}",
            company=company,
            industry=random.choice(INDUSTRIES),
            role=random.choice(ROLES),
            lifecycle_stage=random.choice(LIFECYCLE_STAGES),
            last_engagement=random.choice(LAST_ENGAGEMENTS),
        ))
    return contacts


random.seed(42)  # reproducible results
contacts = generate_contacts(500)

print(f"Generated {len(contacts)} contacts.")
print(f"\nSample contact:")
print(json.dumps(asdict(contacts[0]), indent=2))

## 3 — Build the Personalisation Prompt

Each contact gets a unique prompt that incorporates their CRM attributes. The LLM generates
a short personalised email intro (2–3 sentences).

In [ ]:
# --- Prompt builder ---

SYSTEM_MSG = (
    "You are an expert B2B email copywriter for MarketFlow, an AI-powered marketing "
    "automation platform. Write a personalised email opening paragraph (2-3 sentences) "
    "that feels human, references the recipient's context, and naturally leads into our "
    "Q2 Benchmark Report offer. Do NOT include subject line or sign-off."
)


def build_prompt(contact: Contact) -> List[Dict[str, str]]:
    """Build chat messages for a single contact's personalised email."""
    user_msg = (
        f"Write a personalised email opening for:\n"
        f"- Name: {contact.first_name} {contact.last_name}\n"
        f"- Role: {contact.role} at {contact.company}\n"
        f"- Industry: {contact.industry}\n"
        f"- Lifecycle stage: {contact.lifecycle_stage}\n"
        f"- Last engagement: {contact.last_engagement}\n"
    )
    return [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": user_msg},
    ]


# Preview the prompt for the first contact
sample_prompt = build_prompt(contacts[0])
print("System:\n", sample_prompt[0]["content"])
print("\nUser:\n", sample_prompt[1]["content"])

## 4 — Mock LLM Response Generator

When `USE_MOCK` is True, we generate plausible personalised content locally instead of
calling the API. A small random delay simulates network latency.

In [ ]:
# --- Mock response generator ---

TEMPLATES = [
    (
        "Hi {first_name}, I noticed you recently {engagement} — great to see {company} "
        "staying ahead in the {industry} space. As a {role}, you know that data-driven "
        "decisions separate the top performers from the pack. Our new Q2 Benchmark Report "
        "has some insights I think you'll find particularly relevant."
    ),
    (
        "{first_name}, the {industry} landscape is shifting fast, and {company} is clearly "
        "keeping pace — your recent {engagement} caught our attention. We just released our "
        "Q2 Benchmark Report with fresh data on how marketing leaders like you are "
        "adapting their strategies this quarter."
    ),
    (
        "Great to connect, {first_name}. Leading {stage} engagement at {company} in the "
        "{industry} sector is no small feat. We’ve compiled benchmark data from 500+ "
        "companies in our Q2 report that maps directly to the challenges you’re tackling."
    ),
]


async def mock_llm_call(contact: Contact) -> str:
    """Simulate an LLM call with a templated response and realistic delay."""
    await asyncio.sleep(random.uniform(0.005, 0.02))  # simulate latency
    template = random.choice(TEMPLATES)
    return template.format(
        first_name=contact.first_name,
        engagement=contact.last_engagement,
        company=contact.company,
        industry=contact.industry,
        role=contact.role,
        stage=contact.lifecycle_stage,
    )


print("Mock LLM caller ready.")

## 5 — Live Async LLM Caller

When `USE_MOCK` is False, we use `httpx.AsyncClient` to call the OpenAI chat completions
endpoint. An `asyncio.Semaphore` limits concurrency to avoid rate-limit errors.

In [ ]:
# --- Live async caller ---
import httpx

MAX_CONCURRENCY = 20  # simultaneous requests (adjust per your rate limit tier)
semaphore = asyncio.Semaphore(MAX_CONCURRENCY)


async def live_llm_call(
    client: httpx.AsyncClient,
    contact: Contact,
) -> str:
    """
    Call the OpenAI API for a single contact, respecting the concurrency semaphore.
    """
    messages = build_prompt(contact)
    payload = {
        "model": LLM_MODEL,
        "messages": messages,
        "temperature": 0.7,
        "max_tokens": 150,
    }
    headers = {
        "Authorization": f"Bearer {OPENAI_API_KEY}",
        "Content-Type": "application/json",
    }

    async with semaphore:
        resp = await client.post(
            LLM_BASE_URL,
            json=payload,
            headers=headers,
            timeout=30.0,
        )
        resp.raise_for_status()
        data = resp.json()
        return data["choices"][0]["message"]["content"]


print(f"Live caller ready (max concurrency = {MAX_CONCURRENCY}).")

## 6 — Batch Processing Orchestrator

The orchestrator fans out all 500 calls using `asyncio.gather`, collects results, and
handles errors gracefully by catching exceptions per-contact.

In [ ]:
# --- Batch orchestrator ---

@dataclass
class PersonalisationResult:
    contact_id: int
    email: str
    personalised_intro: str
    success: bool
    error: Optional[str] = None


async def process_contact(
    contact: Contact,
    client: Optional[httpx.AsyncClient] = None,
) -> PersonalisationResult:
    """Process a single contact through the LLM pipeline."""
    try:
        if USE_MOCK:
            intro = await mock_llm_call(contact)
        else:
            intro = await live_llm_call(client, contact)
        return PersonalisationResult(
            contact_id=contact.id,
            email=contact.email,
            personalised_intro=intro,
            success=True,
        )
    except Exception as e:
        return PersonalisationResult(
            contact_id=contact.id,
            email=contact.email,
            personalised_intro="",
            success=False,
            error=str(e),
        )


async def run_batch(contacts: List[Contact]) -> List[PersonalisationResult]:
    """Run personalisation for all contacts concurrently."""
    if USE_MOCK:
        tasks = [process_contact(c) for c in contacts]
    else:
        async with httpx.AsyncClient() as client:
            tasks = [process_contact(c, client) for c in contacts]
            return await asyncio.gather(*tasks)
    return await asyncio.gather(*tasks)


print("Batch orchestrator ready.")

## 7 — Execute the Batch

We run the full 500-contact batch and measure wall-clock time. Compare this to the
estimated sequential time.

In [ ]:
# --- Run the batch ---

start = time.perf_counter()
results = await run_batch(contacts)
elapsed = time.perf_counter() - start

successes = sum(1 for r in results if r.success)
failures = sum(1 for r in results if not r.success)

print(f"\nBatch complete in {elapsed:.2f}s")
print(f"  Successes: {successes}")
print(f"  Failures:  {failures}")
print(f"  Throughput: {len(contacts) / elapsed:.0f} contacts/sec")

# Estimated sequential time (assuming ~2s per call)
est_sequential = len(contacts) * 2.0
print(f"\nEstimated sequential time: {est_sequential:.0f}s ({est_sequential / 60:.1f} min)")
print(f"Speedup: ~{est_sequential / elapsed:.0f}x")

## 8 — Inspect Sample Results

Let’s look at the personalised output for a few contacts to verify quality.

In [ ]:
# --- Inspect results ---

for r in results[:5]:
    contact = contacts[r.contact_id - 1]
    print(f"{'=' * 70}")
    print(f"  Contact #{r.contact_id}: {contact.first_name} {contact.last_name}")
    print(f"  {contact.role} at {contact.company} ({contact.industry})")
    print(f"  Stage: {contact.lifecycle_stage} | Last: {contact.last_engagement}")
    print(f"{'=' * 70}")
    print(r.personalised_intro)
    print()

## 9 — Analytics: Distribution by Lifecycle Stage

In production you would log these results to a data warehouse. Here we compute a quick
breakdown to verify coverage.

In [ ]:
# --- Analytics ---
from collections import Counter

stage_counts = Counter(c.lifecycle_stage for c in contacts)
print("Contacts by lifecycle stage:")
for stage, count in sorted(stage_counts.items(), key=lambda x: -x[1]):
    bar = '#' * (count // 5)
    print(f"  {stage:15s} {count:4d}  {bar}")

print(f"\nAverage intro length: {sum(len(r.personalised_intro) for r in results if r.success) / successes:.0f} chars")

## 10 — Export Results for Email Platform

Finally, we export the results as a JSON file that could be imported into an ESP
(Email Service Provider) like SendGrid, Mailchimp, or HubSpot.

In [ ]:
# --- Export ---

export_data = []
for r in results:
    if r.success:
        contact = contacts[r.contact_id - 1]
        export_data.append({
            "email": r.email,
            "first_name": contact.first_name,
            "last_name": contact.last_name,
            "personalised_intro": r.personalised_intro,
            "company": contact.company,
            "lifecycle_stage": contact.lifecycle_stage,
        })

# Preview (don't write to disk in the notebook)
print(f"Ready to export {len(export_data)} records.")
print(f"\nSample export record:")
print(json.dumps(export_data[0], indent=2))

## Key Takeaways

1. **Async concurrency is essential** — At scale, sequential LLM calls are a non-starter.
   The `asyncio.gather` + `httpx.AsyncClient` pattern gives you 10–20× throughput improvement
   with minimal code complexity.

2. **Semaphore-based throttling** — Use `asyncio.Semaphore` to cap concurrent requests.
   This protects you from rate-limit errors and keeps your API provider happy.

3. **Error isolation** — Wrapping each call in a try/except ensures one failure doesn’t
   crash the entire batch. In production, add retry logic with exponential backoff.

4. **Mock-first development** — Building the full pipeline with mocks lets you validate
   orchestration logic, export formats, and analytics before spending on API calls.

5. **CRM-to-LLM pipeline** — The pattern of extracting contact attributes, building per-contact
   prompts, and fanning out LLM calls is the backbone of AI-powered email personalisation.

---

*Next notebook: [03_design_patterns.ipynb](./03_design_patterns.ipynb) — All 6 agentic design
patterns with martech examples.*